In [2]:
# ============================================================
# U8 — Data Cleaning & Preprocessing (Part 1)
# Complete Solutions for LAB EXERCISES 1–5
# ============================================================

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

print("Setup complete. pandas", pd.__version__)

# ============================================================
# MESSY DATASET
# ============================================================

raw = pd.DataFrame({
    'id':    [1, 2, 3, 4, 5, 6, 7, 7],
    'name':  ['Ana', 'Bo', 'Cy', 'Di', 'Eve', 'Fin', 'Gus', 'Gus'],
    'age':   [30, 25, np.nan, 41, -1, 38, 29, 29],
    'city':  [' Pune ', 'pune', 'DELHI', 'Delhi ', 'Mumbai',
              'bombay', 'Pune.', 'Pune.'],
    'spend': ['120.5', '80.0', '200.2', 'N/A',
              '150.0', '99000', '110.0', '110.0'],
    'date':  ['2024-01-05', '2024-01-06', '2024-01-07',
              '2024-01-08', '2024-01-09', '2024-01-10',
              '2024-01-11', '2024-01-11']
})

# ============================================================
# LAB EXERCISE 1
# Profile the Dataset
# ============================================================

print("\n" + "="*60)
print("LAB EXERCISE 1 — Profile the Dataset")
print("="*60)

df = raw.copy()

# 1. Duplicate rows
print("Duplicate rows:")
print(df.duplicated().sum())

# 2. Missing count per column
print("\nMissing values per column:")
print(df.isna().sum())

# 3. Problems observed:
# - Missing age value (NaN)
# - Age contains -1 as disguised missing value
# - Spend contains 'N/A'
# - Spend stored as text instead of numeric
# - Duplicate row exists
# - Date stored as string
# - City names inconsistent
# - Extreme spend outlier (99000)

# ============================================================
# LAB EXERCISE 2
# Drop vs Impute
# ============================================================

print("\n" + "="*60)
print("LAB EXERCISE 2 — Drop vs Impute")
print("="*60)

ex = raw.copy()

# 1. Unmask missing values
ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')
ex['age'] = ex['age'].replace(-1, np.nan)

# 2a. Dropna version
drop_version = ex.dropna()

# 2b. Median-impute version
impute_version = ex.copy()

impute_version['age'] = (
    impute_version['age']
    .fillna(impute_version['age'].median())
)

impute_version['spend'] = (
    impute_version['spend']
    .fillna(impute_version['spend'].median())
)

# 3. Compare row counts
print("Original rows:", len(ex))
print("Dropna rows  :", len(drop_version))
print("Imputed rows :", len(impute_version))
print("Rows lost by dropping:",
      len(ex) - len(drop_version))

# ============================================================
# Create cleaned df used later
# ============================================================

df = raw.copy()

df['spend'] = pd.to_numeric(df['spend'], errors='coerce')
df['age'] = df['age'].replace(-1, np.nan)

df['age'] = df['age'].fillna(df['age'].median())
df['spend'] = df['spend'].fillna(df['spend'].median())

df = df.drop_duplicates()

df['date'] = pd.to_datetime(df['date'])

# ============================================================
# LAB EXERCISE 3
# Dedupe & Retype
# ============================================================

print("\n" + "="*60)
print("LAB EXERCISE 3 — Dedupe & Retype")
print("="*60)

ex = raw.copy()

# 1. Fix types
ex['spend'] = pd.to_numeric(ex['spend'], errors='coerce')
ex['date'] = pd.to_datetime(ex['date'])

# 2. Drop duplicates
ex = ex.drop_duplicates()

# 3. dtypes + shape
print("Dtypes:")
print(ex.dtypes)

print("\nShape:")
print(ex.shape)

# ============================================================
# LAB EXERCISE 4
# IQR Outlier Detection on Age
# ============================================================

print("\n" + "="*60)
print("LAB EXERCISE 4 — Age Outliers")
print("="*60)

# 1. Q1, Q3, IQR
Q1 = df['age'].quantile(0.25)
Q3 = df['age'].quantile(0.75)
IQR = Q3 - Q1

print("Q1 :", Q1)
print("Q3 :", Q3)
print("IQR:", IQR)

# 2. Bounds
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print("\nLower bound:", lower)
print("Upper bound:", upper)

# 3. Outliers
age_outliers = df[
    (df['age'] < lower) |
    (df['age'] > upper)
]

print("\nAge outliers:")
print(age_outliers)

# ============================================================
# LAB EXERCISE 5
# Clean Messy Text
# ============================================================

print("\n" + "="*60)
print("LAB EXERCISE 5 — Clean Text Categories")
print("="*60)

messy = pd.Series(
    [' London ', 'london', 'LONDON',
     'N.Y.', 'new york ', 'New York'],
    dtype='string'
)

# 1. strip + lower
messy = messy.str.strip().str.lower()

# 2. map n.y. -> new york
messy = messy.replace({
    'n.y.': 'new york'
})

# 3. value_counts
print(messy.value_counts())

# ============================================================
# CLEAN COMPLETE DATASET
# ============================================================

print("\n" + "="*60)
print("FINAL CLEANING PIPELINE")
print("="*60)

clean = raw.copy()

# Missing values
clean['spend'] = pd.to_numeric(
    clean['spend'],
    errors='coerce'
)

clean['age'] = clean['age'].replace(
    -1,
    np.nan
)

clean['age'] = clean['age'].fillna(
    clean['age'].median()
)

clean['spend'] = clean['spend'].fillna(
    clean['spend'].median()
)

# Remove duplicates
clean = clean.drop_duplicates()

# Fix types
clean['date'] = pd.to_datetime(clean['date'])

# Clean city names
clean['city'] = (
    clean['city']
    .astype('string')
    .str.strip()
    .str.lower()
    .str.replace('.', '', regex=False)
    .replace({'bombay': 'mumbai'})
)

print("Final shape:", clean.shape)
print("Missing values:",
      int(clean.isna().sum().sum()))
print("Duplicates:",
      int(clean.duplicated().sum()))

print("\nClean Dataset:")
print(clean)

print("\n" + "="*60)
print("U8 PART-1 LAB COMPLETED")
print("="*60)
print("✓ Data profiling")
print("✓ Missing value handling")
print("✓ Drop vs impute comparison")
print("✓ Duplicate removal")
print("✓ Data type conversion")
print("✓ IQR outlier detection")
print("✓ Text/category standardization")

Setup complete. pandas 2.2.2

LAB EXERCISE 1 — Profile the Dataset
Duplicate rows:
1

Missing values per column:
id       0
name     0
age      1
city     0
spend    0
date     0
dtype: int64

LAB EXERCISE 2 — Drop vs Impute
Original rows: 8
Dropna rows  : 5
Imputed rows : 8
Rows lost by dropping: 3

LAB EXERCISE 3 — Dedupe & Retype
Dtypes:
id                int64
name             object
age             float64
city             object
spend           float64
date     datetime64[ns]
dtype: object

Shape:
(7, 6)

LAB EXERCISE 4 — Age Outliers
Q1 : 29.25
Q3 : 34.0
IQR: 4.75

Lower bound: 22.125
Upper bound: 41.125

Age outliers:
Empty DataFrame
Columns: [id, name, age, city, spend, date]
Index: []

LAB EXERCISE 5 — Clean Text Categories
london      3
new york    3
Name: count, dtype: Int64

FINAL CLEANING PIPELINE
Final shape: (7, 6)
Missing values: 0
Duplicates: 0

Clean Dataset:
   id name   age    city    spend       date
0   1  Ana  30.0    pune    120.5 2024-01-05
1   2   Bo  25.0   